# 01 - Dataset exploration and profiling
UCI *Movie* dataset (id 132, Wiederhold). Raw files are read without modification to understand formats, encodings, schema, size and quality.

In [1]:
import sys, os, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 90)
def show(f, w=900): display(Image(filename=str(ROOT / 'outputs' / 'plots' / f), width=w))

In [2]:
from src.data_loading.html_table_loader import file_inventory, read_table
pd.DataFrame(file_inventory())

,file,bytes,format,encoding,data_rows,header_rows
0,actors.html,811933,html,utf-8,6818.0,31.0
1,adds.txt,162138,txt,ascii,NaN,NaN
2,Awardtypes.doc,39936,doc,binary (MS Word),NaN,NaN
3,awtypes.html,3336,html,utf-8,35.0,1.0
4,casts.html,4358587,html,latin-1,46197.0,2616.0
5,keys.txt,892,txt,ascii,NaN,NaN
6,locales.html,1139,html,utf-8,14.0,1.0
7,main.html,2094935,html,utf-8,11477.0,2957.0
8,movies.macros,4509,macros,ascii,NaN,NaN
9,people.html,355915,html,utf-8,3311.0,22.0


## Raw record format
Each record is one `<tr><td> ... <td>|` line; the number of cells per row reveals malformed records.

In [3]:
main = read_table('main.html')
print(len(main), 'MAIN rows')
print(pd.Series([len(r.fields) for r in main]).value_counts().sort_index())
for r in main[:5]: print(r.fields)

11477 MAIN rows
0         4
2         1
4         1
5         1
6        30
7        50
8        77
9       123
10      448
11    10606
12      122
13        4
14        2
15        4
16        3
17        1
Name: count, dtype: int64
['H1', 'T:Always Tell Your Wife', '1922', 'D:Se.Hicks', 'P:Lasky', 'St:Famous, SD:*', 'sbw', 'Dram', 'aw', 'lc', 'CoD(Hitchcock)']
['H2', 'T:Number Thirteen', '1922', 'D:Hitchcock', 'P:Hitchcock', 'St:Islington, SD:Famous', 'sbw', '', '', '', 'Nt(unfinished)']
['H3', 'T:Woman to Woman', '1922', 'D:Hitchcock', 'P:Balcon', 'St:B-S-F, SD:Wardour', 'sbw', 'Dram', '', 'England', 'CoD(Cutts) Er(same as GCt27, 1926?)']
['H4', 'T:The Passionate Adventure', '1924', 'D:Hitchcock', 'P:Balcon', 'St:Gainsborough, SD:GaumontD', 'sbw', '', '', '', 'CoD(Cutts)']
['H5', 'T:The Blackguard', '1925', 'D:Hitchcock', 'P:Balcon', 'St:UFA, SD:Wardour', 'sbw', '', '', '', 'CoD(Cutts)']


In [4]:
from src.preprocessing.pipeline import raw_frames, quality_summary, PLACEHOLDERS
raws = raw_frames()
{k: v.shape for k, v in raws.items()}

{'MAIN': (11477, 12),
 'CASTS': (46197, 8),
 'ACTORS': (6818, 12),
 'PEOPLE': (3311, 11),
 'REMAKES': (1277, 8),
 'STUDIOS': (207, 10)}

In [5]:
quality_summary('MAIN', raws['MAIN'].drop(columns='_n_fields'), PLACEHOLDERS['MAIN'])

,table,attribute,n_records,missing_or_placeholder,missing_pct,n_unique,most_frequent,most_frequent_count
0,MAIN,film_id,11477,6,0.05,11418,,6
1,MAIN,title_raw,11477,5,0.04,10773,T:Romeo and Juliet,14
2,MAIN,year_raw,11477,22,0.19,148,1992,277
3,MAIN,director_raw,11477,6,0.05,3042,D:Hitchcock,79
4,MAIN,producer_raw,11477,6560,57.16,2635,PN:,5487
5,MAIN,studio_raw,11477,5623,48.99,1357,SU:,5609
6,MAIN,process_raw,11477,4723,41.15,246,prc,4012
7,MAIN,category_raw,11477,3168,27.60,445,Ctxx,2099
8,MAIN,awards_raw,11477,7999,69.70,532,aw,4858
9,MAIN,locale_raw,11477,6458,56.27,2996,lc,4022


In [6]:
pd.read_csv('outputs/data_dictionary.csv')

,table,attribute,type,description
0,MAIN,film_id,nominal (key),Internally generated id = director code + sequence number
1,MAIN,title,text,"Film title, prefixed T:/Tn:; not unique"
2,MAIN,year,numeric (interval),Release year; '19xx' = unknown
3,MAIN,director,nominal (FK->PEOPLE),"Standardised director id-name, prefixed D:"
4,MAIN,producers,"nominal, multi-valued",P:(in PEOPLE) / PN:(name) / PU:(unknown) / P:(none)
5,MAIN,studios,"nominal, multi-valued","St:/S: studio, SU: unknown, SL: location, SD: distributor"
6,MAIN,process,nominal (coded),"bws/sbw silent b/w, bnw, col, \Tcol etc., cld colourised, prc unknown"
7,MAIN,category,"nominal, multi-valued","Genre codes (Dram, Comd, Susp ...); Ctxx = uncategorised"
8,MAIN,awards,"nominal, multi-valued","H..H**** Halliwell stars, AA/AAN Oscar, CG, RE, NFR ...; aw unknown"
9,MAIN,locale,"text, hierarchical",Where the film is set; lc = unknown


In [7]:
print(open('outputs/dataset_overview.txt', encoding='utf-8').read())

UCI MOVIE DATASET - OVERVIEW
Source : UCI Machine Learning Repository, dataset id 132 'Movie'
URL    : https://archive.ics.uci.edu/ml/datasets/Movie (zip: /static/public/132/movie.zip)
Creator: Gio Wiederhold (Stanford); DOI 10.24432/C5SW2R; licence CC BY 4.0
Type   : Multivariate, relational; 6 linked HTML tables + documentation (doc.html)

FILES
------------------------------------------------------------
          file   bytes format         encoding  data_rows  header_rows
   actors.html  811933   html            utf-8     6818.0         31.0
      adds.txt  162138    txt            ascii        NaN          NaN
Awardtypes.doc   39936    doc binary (MS Word)        NaN          NaN
  awtypes.html    3336   html            utf-8       35.0          1.0
    casts.html 4358587   html          latin-1    46197.0       2616.0
      keys.txt     892    txt            ascii        NaN          NaN
  locales.html    1139   html            utf-8       14.0          1.0
     main.html 209493